In [26]:
import ROOT
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
import math
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali


ROOT.gROOT.SetBatch(True)

In [27]:
file_path = "esperienza5.xlsx"

sheets = {
    'Dati': 'Dati',
    'Manuele': 'Manuele',
    'Alessio': 'Alessio',
    'Emanuele': 'Emanuele',
    'Michele': 'Michele'
}

def load_data(sheet_name):
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=0)
        theta = pd.to_numeric(df.iloc[:, 0], errors='coerce')
        print(theta)
        theta = theta.dropna()
        
        theta_val = theta.values.astype(np.float64)
        
        return theta_val
    except Exception as e:
        print(f"Errore nel caricamento del foglio {sheet_name}: {e}")
        return None, None


def IQR(dist):
    return np.percentile(dist, 75) - np.percentile(dist, 25)

In [ ]:
root_objects = []
def make_hist(nomi, title, filename):
    c = ROOT.TCanvas(f"c_{filename}", title, 800, 600)
    c.SetGrid()
    

    for name in nomi:
        if name in data:
            theta = data[name]

            gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(theta), max(theta), 3)
            gaussian_fit.SetParNames("const", "mu", "sigma")
            gaussian_fit.SetLineColor(ROOT.kRed + 1)
            gaussian_fit.SetLineWidth(3)


            delta_bin = 2*IQR(theta)/ (len(theta))**(1/3)
            n_bin= (np.max(theta) - np.min(theta))/delta_bin
            n_bin = int(np.round(n_bin))

            if name == "Manuele":
                hist = ROOT.TH1F("hist", f"{title}; θ_B; conteggi", 15, min(theta), max(theta))
                print(theta)
            else:
                 hist = ROOT.TH1F("hist", f"{title}; θ_B; conteggi", n_bin, min(theta), max(theta))
            for val in theta:
                hist.Fill(val)
                leg = ROOT.TLegend(0.15, 0.70, 0.40, 0.85)
                leg.SetBorderSize(1)
                leg.SetFillStyle(1001)

        gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
            
        hist.SetLineColor(ROOT.kAzure + 3)
        hist.SetLineWidth(2)
        hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
        hist.GetXaxis().SetTitle("θ_B")
        hist.GetYaxis().SetTitle("conteggi")
        hist.SetTitle(f"{title}")

        r = hist.Fit(gaussian_fit, "S")

        fitStatus = r.Status()
        print(f"Fit status: {fitStatus}")

        parameters = gaussian_fit.GetParameters()
        errors = gaussian_fit.GetParErrors()

        chi2 = gaussian_fit.GetChisquare()
        '''
        print("Parametri stimati:")
        print(f"const = {parameters[0]} ± {errors[0]}")
        print(f"#mu = {parameters[1]} ± {errors[1]}")
        print(f"#sigma = {parameters[2]} ± {errors[2]}")
        print(f"Chi2 = {chi2}")
        print("\n")
        print("--------------------------------------------------------------")
        print("\n")
        '''
        root_objects.append(hist)

    hist.Draw()

    '''
    leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
    leg2.AddEntry(hist, "Residui", "f")
    leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
    leg2.Draw()
    '''

    c.SaveAs(filename)


In [29]:
data = {}
for key, s_name in sheets.items():
    theta = load_data(s_name)
    data[key] = (theta) 


0      56.370000
1      57.408333
2      57.200000
3      56.891667
4      57.200000
         ...    
100    56.733333
101    56.558333
102    56.416667
103    56.541667
104    56.500000
Name: θ_Bi, Length: 105, dtype: float64
0     56.370000
1     57.408333
2     57.200000
3     56.891667
4     57.200000
5     56.900000
6     57.375000
7     57.300000
8     57.366667
9     57.291667
10    57.200000
11    57.616667
12    57.450000
13    57.125000
14    57.250000
15    57.450000
16    57.116667
17    56.700000
18    56.450000
19    56.891667
20    56.583333
21    56.450000
22    56.741667
23    57.083333
24    57.075000
25    57.550000
26    56.783333
27    56.766667
28    57.325000
29    56.700000
30    56.700000
31    56.450000
32    56.291667
33    56.450000
34    57.200000
35    57.450000
36    56.575000
37    56.450000
38    56.625000
39    56.700000
40    56.733333
41    56.783333
42    56.566667
43    56.566667
44    56.533333
45    56.533333
46    56.333333
47    56.650000
48   

In [30]:
make_hist(['Dati'], 'Istogramma con tutti i dati', 'hist_totalData.png')
make_hist(['Manuele'], 'Istogramma con i dati di Manuele', 'histManuData.png')
make_hist(['Alessio'], 'Istogramma con i dati di Alessio', 'histAleData.png')
make_hist(['Emanuele'], 'Istogramma con i dati di Emanuele', 'histStroiliData.png')


Fit status: 0
[56.37       57.40833333 57.2        56.89166667 57.2        56.9
 57.375      57.3        57.36666667 57.29166667 57.2        57.61666667
 57.45       57.125      57.25       57.45       57.11666667 56.7
 56.45       56.89166667 56.58333333 56.45       56.74166667 57.08333333
 57.075      57.55       56.78333333 56.76666667 57.325      56.7
 56.7        56.45       56.29166667 56.45       57.2        57.45
 56.575      56.45       56.625      56.7        56.73333333 56.78333333
 56.56666667 56.56666667 56.53333333 56.53333333 56.33333333 56.65
 56.525      56.45       56.50833333 56.56666667 56.61666667 56.4
 56.51666667 57.01666667 57.03333333 56.36666667 56.7        57.1       ]
Fit status: 0
Fit status: 0
Fit status: 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      11.8569
NDf                       =            5
Edm                       =  8.43337e-07
NCalls                    =           74
const             

Info in <TCanvas::Print>: png file hist_totalData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histManuData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histAleData.png has been created
Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Info in <TCanvas::Print>: png file histStroiliData.png has been created
